3) Map-Refine 요약

In [ ]:
# [목적] LangSmith 프롬프트와 ChatOpenAI 모델을 준비해 Map-Refine 요약 환경을 구성합니다.
# 환경 변수와 실행 추적을 설정한 뒤 문서별 초벌 요약에 사용할 모델과 프롬프트를 불러옵니다.
# 여기서 준비한 객체는 이후 Map 체인을 만들고 전체 요약 과정을 추적하는 데 사용합니다.
from dotenv import load_dotenv
from langchain_teddynote import logging
from langsmith import Client
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


# .env 파일에 저장된 OpenAI·LangSmith API 키 등의 환경 변수를 불러옵니다.
load_dotenv()

# 이후 LangChain 실행 과정을 LangSmith의 'Summary' 프로젝트에서 추적합니다.
logging.langsmith("Summary")


# ChatOpenAI는 OpenAI 채팅 모델을 LangChain 체인에서 사용할 수 있게 감싼 객체입니다.
map_llm = ChatOpenAI(
    # 0에 가까울수록 무작위성을 줄여 페이지별 요약을 일관되게 생성합니다.
    temperature=0,
    model_name="gpt-4o-mini",
)


# Client는 LangSmith Hub에 저장된 프롬프트를 조회하는 연결 객체입니다.
client = Client()

# 문서 한 개를 초벌 요약하는 공개 Map 프롬프트를 가져옵니다.
map_summary = client.pull_prompt(
    "teddynote/map-summary-prompt",
    dangerously_pull_public_prompt=True,
)

# 프롬프트 내용 확인
map_summary.pretty_print()

In [ ]:
# [목적] PyPDFLoader로 PDF 일부 페이지를 불러와 Map-Refine 요약의 입력 문서를 준비합니다.
# PDF를 페이지별 Document 객체로 읽은 뒤 예제에서 사용할 4~8페이지만 선택합니다.
# 선택된 docs는 Map 단계에서 각각 요약되고 Refine 단계에서 순서대로 통합됩니다.
from langchain_community.document_loaders import PyPDFLoader

# PyPDFLoader는 각 PDF 페이지를 본문과 메타데이터가 담긴 Document로 변환합니다.
loader = PyPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()

docs = docs[3:8]

print(f"총 페이지수: {len(docs)}")

In [ ]:
# [목적] Map 프롬프트와 모델을 연결해 각 문서의 초벌 요약을 만드는 체인을 구성합니다.
# 파이프(|) 순서대로 프롬프트 구성, 모델 호출, 문자열 변환이 실행됩니다.
# StrOutputParser가 모델 응답을 일반 문자열로 바꿔 이후 Refine 단계에서 사용하기 쉽게 만듭니다.
map_chain = map_summary | map_llm | StrOutputParser()

In [ ]:
# [목적] 첫 번째 PDF 페이지에 Map 체인을 시험 실행해 초벌 요약 결과를 확인합니다.
# 문서와 출력 언어를 프롬프트 변수로 전달하고 invoke로 한 건의 요약을 생성합니다.
# 전체 페이지를 일괄 처리하기 전에 체인과 입력 형식이 올바른지 점검하는 과정입니다.
print(map_chain.invoke({"documents": docs[0], "language": "Korean"}))

In [ ]:
# [목적] 여러 문서를 Map 체인에 일괄 전달할 입력 목록으로 변환합니다.
# 각 Document를 documents 키에 넣고 모든 결과의 출력 언어를 한국어로 지정합니다.
# 이렇게 만든 input_doc은 다음 셀의 batch 실행에서 페이지별 입력으로 사용합니다.
# 리스트 컴프리헨션은 docs의 각 문서를 같은 구조의 딕셔너리로 바꿉니다.
input_doc = [{"documents": doc, "language": "Korean"} for doc in docs]
input_doc

In [ ]:
# [목적] Map 체인을 모든 페이지 입력에 일괄 실행해 초벌 요약 목록을 확인합니다.
# batch는 input_doc의 각 항목에 같은 체인을 적용하고 입력 순서대로 결과를 반환합니다.
# 이 결과들은 이후 Refine 과정에서 앞 요약에 다음 요약을 차례로 반영할 재료가 됩니다.
print(map_chain.batch(input_doc))

In [ ]:
# [목적] 기존 요약에 다음 요약을 반영하기 위한 Refine 전용 프롬프트를 준비합니다.
# LangSmith Hub에서 공개 프롬프트를 가져온 뒤 필요한 입력 변수와 지시문을 출력합니다.
# 이 프롬프트는 previous_summary와 current_summary를 받아 누적 요약을 갱신합니다.
refine_prompt = client.pull_prompt(
    "teddynote/refine-prompt",
    dangerously_pull_public_prompt=True,
)

refine_prompt.pretty_print()

In [ ]:
# [목적] Refine 프롬프트와 모델을 연결해 누적 요약을 갱신하는 체인을 구성합니다.
# 이전 요약과 새 요약을 프롬프트에 넣고 모델이 보완한 결과를 문자열로 반환합니다.
# 이 체인을 문서 순서대로 반복 실행하면 전체 내용을 반영한 최종 요약이 만들어집니다.
# Refine 단계에서 사용할 OpenAI 채팅 모델을 생성합니다.
refine_llm = ChatOpenAI(  # refine llm 생성
    temperature=0,
    model_name="gpt-4o-mini",
)

# Refine 프롬프트 → 모델 → 문자열 변환 순서로 실행되는 체인입니다.
refine_chain = refine_prompt | refine_llm | StrOutputParser()

In [ ]:
# [목적] 문서별 Map 요약을 차례로 보완하는 전체 Map-Refine 과정을 하나의 체인으로 만듭니다.
# 모든 문서를 먼저 요약한 뒤 첫 요약부터 시작해 다음 요약을 하나씩 누적 반영합니다.
# 반복 작업을 함수로 묶어 같은 형식의 다른 문서 목록에도 invoke 한 번으로 적용할 수 있습니다.
from langchain_core.runnables import chain

# @chain은 일반 Python 함수를 invoke로 실행할 수 있는 LangChain Runnable로 변환합니다.
@chain
def map_refine_chain(docs):

    map_summary = client.pull_prompt(
        "teddynote/map-summary-prompt",
        dangerously_pull_public_prompt=True,
    )

    map_chain = (
        map_summary
        | ChatOpenAI(
            model_name="gpt-4o-mini",
            temperature=0,
        )
        | StrOutputParser()
    )

    input_doc = [
        {"documents": doc.page_content, "language": "Korean"}
        for doc in docs
    ]

    # 각 페이지를 한꺼번에 처리해 순서가 유지된 초벌 요약 목록을 만듭니다.
    doc_summaries = map_chain.batch(input_doc)

    refine_prompt = client.pull_prompt(
        "teddynote/refine-prompt",
        dangerously_pull_public_prompt=True,
    )

    refine_llm = ChatOpenAI(
        model_name="gpt-4o-mini",
        temperature=0,
        # StreamingCallback은 갱신되는 요약을 생성되는 순서대로 Notebook에 표시합니다.
        callbacks=[StreamingCallback()],
        streaming=True,
    )

    refine_chain = refine_prompt | refine_llm | StrOutputParser()

    # 첫 번째 초벌 요약을 누적 요약의 시작값으로 사용합니다.
    previous_summary = doc_summaries[0]

    # 두 번째 요약부터 하나씩 반영하며 previous_summary를 최신 결과로 교체합니다.
    for current_summary in doc_summaries[1:]:

        previous_summary = refine_chain.invoke(
            {
                "previous_summary": previous_summary,
                "current_summary": current_summary,
                "language": "Korean",
            }
        )

        print("\n\n----------------\n\n")

    return previous_summary

In [ ]:
# [목적] 완성된 Map-Refine 체인을 PDF 문서 목록에 실행해 최종 누적 요약을 생성합니다.
# 콜백을 먼저 준비한 뒤 docs를 체인에 전달하면 Refine 결과가 실시간으로 표시됩니다.
# 모든 페이지가 반영된 마지막 요약은 refined_summary에 저장해 이후 작업에 사용할 수 있습니다.
from langchain_teddynote.callbacks import StreamingCallback

refined_summary = map_refine_chain.invoke(docs)